# 🤖 Natural-Language-to-SQL Analyst Assistant for Trading Data

**AI Engineer Track**  |  Difficulty: **Medium**  |  Domain: **Trading / Transaction Analytics**

> 💯 Built with 100% free tools — no paid API keys required. Uses a free `call_llm()` helper that
> tries **Cerebras → Groq → local Ollama** in that order, so this notebook costs $0 to run.

---

## 🧩 Problem Statement

Trade desk analysts need instant answers about trading activity but shouldn't have to write SQL, and firms are often not permitted to send trade data to a third-party API. Build a fully local NL-to-SQL assistant using our free call_llm() helper, so trading data never leaves the machine and the tool costs $0 to run.

## 📁 Dataset

**Any trade/transaction blotter (trade_id, symbol, side, quantity, price, timestamp, trader)**

Source: [https://www.kaggle.com/datasets/nikhilkohli/us-stock-market-data-60-extracted-features](https://www.kaggle.com/datasets/nikhilkohli/us-stock-market-data-60-extracted-features)

⚠️ **Note:** If the real dataset file isn't uploaded to this Colab session, the code below
automatically generates a small realistic sample dataset with the same structure — so every cell
still runs successfully end-to-end even before you upload the real data.


## 🛠️ Tools Used

`Python 3 | Cerebras/Groq (free tiers) + Ollama fallback via local HTTP (all free, no paid API) | SQLite | pandas`

## 🔑 Before You Run

This notebook will ask for a **free Cerebras API key** and a **free Groq API key** (both have
generous free tiers, no credit card needed). You can get keys at:
- Cerebras: https://cloud.cerebras.ai
- Groq: https://console.groq.com/keys

If you skip both (just press Enter), it'll try to use a local Ollama server instead — that only
works if you have Ollama running on your own machine, not inside Colab.

---

### ⚠️ Disclaimer
This notebook is for educational / portfolio purposes only. It does not constitute financial,
legal, or investment advice.

---


In [3]:
!pip install openai requests pandas --break-system-packages

In [4]:
import os
import sqlite3
from getpass import getpass
import requests
from openai import OpenAI
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# STEP 0: set up our 3 free AI options - cerebras first, then groq, then local ollama
# we ask for the api keys once, then build one call_llm() function that tries all 3
# ---------------------------------------------------------
if not os.environ.get("CEREBRAS_API_KEY"):
    os.environ["CEREBRAS_API_KEY"] = getpass("Enter your Cerebras API key (press enter to skip): ")
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key (press enter to skip): ")

cerebras_client = OpenAI(api_key=os.environ.get("CEREBRAS_API_KEY", ""), base_url="https://api.cerebras.ai/v1")
groq_client = OpenAI(api_key=os.environ.get("GROQ_API_KEY", ""), base_url="https://api.groq.com/openai/v1")
OLLAMA_URL = "http://localhost:11434/api/generate"

def call_llm(prompt, system=None):
    # if there's a system instruction, we just stick it on top of the prompt
    # since we're keeping this simple and not building a full messages list
    full_prompt = f"{system}\n\n{prompt}" if system else prompt
    messages = [{"role": "user", "content": full_prompt}]

    # try cerebras first, it's free and fast
    try:
        response = cerebras_client.chat.completions.create(model="gpt-oss-120b", messages=messages)
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("cerebras failed:", e)

    # try groq next
    try:
        response = groq_client.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("groq failed:", e)

    # last resort - ollama running locally, called with a plain http request
    # (no extra "ollama" package needed, just "requests" which colab already has)
    try:
        resp = requests.post(OLLAMA_URL, json={"model": "llama3", "prompt": full_prompt, "stream": False})
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        print("ollama failed too:", e)
        return "AI call failed, all 3 options did not work"


conn = sqlite3.connect("trading.db")






In [5]:

# ---------------------------------------------------------
# STEP 1: load the trade blotter, if it's missing just fake up some sample trades
# ---------------------------------------------------------
def load_trades(path="trade_blotter.csv"):
    if os.path.exists(path):
        return pd.read_csv(path, parse_dates=["timestamp"])
    print(f"couldn't find {path}, making some sample trades instead so the code runs")
    np.random.seed(1)
    n = 300
    return pd.DataFrame({
        "trade_id": range(1, n + 1),
        "symbol": np.random.choice(["AAPL", "MSFT", "TSLA", "JPM"], n),
        "side": np.random.choice(["buy", "sell"], n),
        "quantity": np.random.randint(10, 500, n),
        "price": np.round(np.random.uniform(100, 400, n), 2),
        "timestamp": pd.date_range("2025-06-01", periods=n, freq="h"),
        "trader": np.random.choice(["trader_a", "trader_b", "trader_c"], n),
    })

df = load_trades()
df.to_sql("trades", conn, if_exists="replace", index=False)



couldn't find trade_blotter.csv, making some sample trades instead so the code runs


300

In [6]:
# ---------------------------------------------------------
# STEP 2: look at our own database and figure out what tables/columns exist
# this way the tool works even if someone changes the column names later
# ---------------------------------------------------------
def get_schema(connection):
    cur = connection.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]
    lines = []
    for t in tables:
        cur.execute(f"PRAGMA table_info({t})")
        cols = [row[1] for row in cur.fetchall()]
        lines.append(f"Table {t}: {', '.join(cols)}")
    return "\n".join(lines)

SCHEMA = get_schema(conn)

In [7]:
# ---------------------------------------------------------
# STEP 3: ask our free call_llm() to write the sql for us based on a plain english question
# ---------------------------------------------------------
def generate_sql(question: str) -> str:
    prompt = f"""Here's the database schema:
{SCHEMA}
Write ONE SQLite SELECT query (just the query, no explanation, no markdown) that answers:
"{question}" """
    sql = call_llm(prompt).strip().strip("`").replace("sql\n", "")
    return sql


In [8]:
# ---------------------------------------------------------
# STEP 4: never trust ai-generated sql blindly, check it's a safe SELECT only
# ---------------------------------------------------------
FORBIDDEN = ["insert", "update", "delete", "drop", "alter", "truncate"]
def validate_sql(sql: str) -> bool:
    lowered = sql.lower()
    if not lowered.strip().startswith("select"):
        return False
    return not any(w in lowered for w in FORBIDDEN)




In [9]:
# ---------------------------------------------------------
# STEP 5: also ask it to explain the query in plain english for the trader
# ---------------------------------------------------------
def explain_query(sql: str) -> str:
    prompt = f"Explain this SQL query in 2 simple sentences a non-technical trader would understand:\n{sql}"
    return call_llm(prompt)

In [10]:

# ---------------------------------------------------------
# STEP 6: put it all together
# ---------------------------------------------------------
def query_assistant(question: str):
    sql = generate_sql(question)
    if not validate_sql(sql):
        raise ValueError(f"this query looked unsafe so we didn't run it: {sql}")
    result = pd.read_sql_query(sql, conn)
    explanation = explain_query(sql)
    print(f"SQL:\n{sql}\n\nExplanation:\n{explanation}\n\nResult:\n{result.head()}")
    return sql, result, explanation

if __name__ == "__main__":
    query_assistant("Which trader had the largest total buy volume in AAPL last month?")

cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
SQL:
SELECT trader
FROM trades
WHERE symbol = 'AAPL'
  AND side = 'buy'
  AND timestamp >= date('now','start of month','-1 month')
  AND timestamp < date('now','start of month')
GROUP BY trader
ORDER BY SUM(quantity) DESC
LIMIT 1;

Explanation:
It looks at all the “buy” trades for Apple (AAPL) that happened during the previous calendar month, adds up how many shares each trader bought, and then picks the trader with the highest total. In short, it tells you who was the top Apple buyer last month.

Result:
Empty DataFrame
Columns: [trader]
Index: []
